# Original Promptriever corpus encoding


In [ ]:
import gc
import glob
import hashlib
import json
import os
import time
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import login, snapshot_download
from peft import PeftModel
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig


BASE_MODEL = "meta-llama/Llama-2-7b-hf"
PROMPTRIEVER_ADAPTER_REPO = "samaya-ai/promptriever-llama2-7b-v1"

MAIN_INPUT_PATH = "/kaggle/input/datasets/sukiss/prmptr/tevatron_ru_promptriever_train (2).jsonl"
TEST_CHUNKS_PATH = "/kaggle/input/datasets/sukiss/test-dataset/chunks_testset.jsonl"

OUT_DIR = "/kaggle/working/promptriever_original_encoding"
ADAPTER_CACHE_DIR = "/kaggle/working/promptriever_original_adapter"

BATCH_DOCS = 16
PASSAGE_MAX_LEN = 128
DTYPE_SAVE = np.float16

HF_TOKEN = os.environ.get("HF_TOKEN", "")


def sha1_id(text: str) -> str:
    return hashlib.sha1(text.strip().encode("utf-8")).hexdigest()


def write_corpus_jsonl(docid_to_text: dict, out_jsonl: str):
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for did, text in docid_to_text.items():
            f.write(json.dumps({"docid": did, "text": text}, ensure_ascii=False) + "\n")


def parse_docs_from_main(path: str):
    docs = {}
    stats = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            stats["rows_seen"] += 1
            try:
                item = json.loads(line)
            except Exception:
                stats["bad_json"] += 1
                continue

            for key in ["positive_passages", "negative_passages"]:
                for passage in item.get(key, []) or []:
                    text = (passage.get("text") or "").strip()
                    if not text:
                        stats["empty_text"] += 1
                        continue

                    did = str(passage.get("docid") or "") or sha1_id(text)
                    if did in docs:
                        stats["dup_docid"] += 1
                    else:
                        docs[did] = text
                        stats["docs_added"] += 1

    return docs, stats


def parse_docs_from_chunks(path: str):
    docs = {}
    stats = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            stats["rows_seen"] += 1
            item = json.loads(line)

            texts = [(item.get("positive") or "").strip()]
            texts += [(x or "").strip() for x in (item.get("original_negs") or [])[:2]]
            texts += [(x or "").strip() for x in (item.get("generated_negs") or [])[:3]]

            for text in texts:
                if not text:
                    stats["empty_text"] += 1
                    continue

                did = sha1_id(text)
                if did in docs:
                    stats["dup_docid"] += 1
                else:
                    docs[did] = text
                    stats["docs_added"] += 1

    return docs, stats


def resolve_original_adapter():
    if os.path.exists(os.path.join(ADAPTER_CACHE_DIR, "adapter_config.json")):
        return ADAPTER_CACHE_DIR

    return snapshot_download(
        repo_id=PROMPTRIEVER_ADAPTER_REPO,
        local_dir=ADAPTER_CACHE_DIR,
        local_dir_use_symlinks=False,
        token=HF_TOKEN if HF_TOKEN else None,
    )


def load_model(adapter_dir: str):
    if HF_TOKEN:
        login(token=HF_TOKEN)

    gc.collect()
    torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            adapter_dir,
            use_fast=False,
            token=HF_TOKEN if HF_TOKEN else None,
        )
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(
            BASE_MODEL,
            use_fast=False,
            token=HF_TOKEN if HF_TOKEN else None,
        )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base = AutoModel.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    base.config.use_cache = False

    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    model.eval()
    print("Loaded original Promptriever:", PROMPTRIEVER_ADAPTER_REPO)
    print("Device:", next(model.parameters()).device)

    return tokenizer, model


def add_eos(tokenizer, texts):
    return [text + tokenizer.eos_token for text in texts]


def eos_pool(last_hidden_state, attention_mask):
    lengths = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
    return last_hidden_state[idx, lengths]


@torch.inference_mode()
def encode_texts(tokenizer, model, texts, max_len):
    batch = tokenizer(
        add_eos(tokenizer, texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(model.device) for k, v in batch.items()}

    out = model(**batch)
    emb = eos_pool(out.last_hidden_state, batch["attention_mask"])
    return F.normalize(emb, p=2, dim=-1)


def encode_corpus(tokenizer, model, docid_to_text: dict, name: str):
    out_jsonl = os.path.join(OUT_DIR, f"{name}_corpus.jsonl")
    out_docids = os.path.join(OUT_DIR, f"{name}_docids.txt")
    out_emb = os.path.join(OUT_DIR, f"{name}_embeddings.npy")

    docids = list(docid_to_text.keys())
    texts = [docid_to_text[d] for d in docids]
    if not texts:
        raise ValueError(f"[{name}] corpus is empty")

    print(f"\n[{name}] docs:", len(docids))
    write_corpus_jsonl(docid_to_text, out_jsonl)

    first = encode_texts(tokenizer, model, texts[:1], PASSAGE_MAX_LEN).detach().cpu().numpy().astype(DTYPE_SAVE)
    dim = first.shape[1]

    emb = np.memmap(out_emb, dtype=DTYPE_SAVE, mode="w+", shape=(len(docids), dim))
    emb[0:1] = first

    start = time.time()
    for i in range(1, len(texts), BATCH_DOCS):
        j = min(len(texts), i + BATCH_DOCS)
        batch_emb = encode_texts(tokenizer, model, texts[i:j], PASSAGE_MAX_LEN).detach().cpu().numpy().astype(DTYPE_SAVE)
        emb[i:j] = batch_emb
        if j % 500 == 0:
            print(f"[{name}] encoded {j}/{len(texts)}")

    emb.flush()

    with open(out_docids, "w", encoding="utf-8") as f:
        for did in docids:
            f.write(did + "\n")

    print(f"[{name}] saved:")
    print(" corpus:", out_jsonl, "MB", round(os.path.getsize(out_jsonl) / 1024 / 1024, 3))
    print(" docids:", out_docids, "MB", round(os.path.getsize(out_docids) / 1024 / 1024, 3))
    print(" emb   :", out_emb, "MB", round(os.path.getsize(out_emb) / 1024 / 1024, 3), "dim", dim)
    print(" time_s:", round(time.time() - start, 1))


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    assert os.path.exists(MAIN_INPUT_PATH), f"Missing MAIN_INPUT_PATH: {MAIN_INPUT_PATH}"
    assert os.path.exists(TEST_CHUNKS_PATH), f"Missing TEST_CHUNKS_PATH: {TEST_CHUNKS_PATH}"

    adapter_dir = resolve_original_adapter()
    tokenizer, model = load_model(adapter_dir)

    main_docs, main_stats = parse_docs_from_main(MAIN_INPUT_PATH)
    print("Main stats:", main_stats)

    test_docs, test_stats = parse_docs_from_chunks(TEST_CHUNKS_PATH)
    print("Test stats:", test_stats)

    encode_corpus(tokenizer, model, main_docs, "main")
    encode_corpus(tokenizer, model, test_docs, "test")

    print("\nDONE. Outputs in:", OUT_DIR)
    print(sorted(os.listdir(OUT_DIR)))


if __name__ == "__main__":
    main()
